In [1]:
import pandas as pd 
import re
import html
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

tqdm.pandas()
print("libraries imported for news preprocessing")

libraries imported for news preprocessing


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv('rawData/dataNews.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         234 non-null    object
 1   publish_date  234 non-null    object
 2   content       234 non-null    object
dtypes: object(3)
memory usage: 5.6+ KB


In [3]:
intial_len = len(df)
df.drop_duplicates(subset=['content'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"total news: {len(df)} (deleted {intial_len - len(df)} duplicate)")

total news: 234 (deleted 0 duplicate)


In [4]:
def clean_news_bertopic(text):
    if not isinstance(text, str):
        return ""
    
    # hapus html
    text = html.unescape(text)

    # hapus elemen non berita
    patterns = [
        r'(?i)^baca juga[:\-\s].*$',
        r'(?i)^simak juga[:\-\s].*$',
        r'(?i)^lihat juga[:\-\s].*$',
        r'(?i)^advertisement.*$',
        r'(?i)^scroll to continue.*$',
        r'(?i)^video[:\-\s].*$',
        r'(?i)^foto[:\-\s].*$'
    ]

    for p in patterns:
        text = re.sub(p, '', text, flags=re.MULTILINE)
    
    # hapus label inline
    text = re.sub(
        r'(?i)SCROLL TO CONTINUE WITH CONTENT|ADVERTISEMENT|BACA JUGA:|SIMAK JUGA:|VIDEO:|FOTO:',
        '',
        text
    )

    # hapus url
    text = re.sub(r'http\S+|www\S+', '', text)

    # normalisasi kutipan
    text = re.sub(r'[“”]', '"', text)

    # hapus karakter non alfanumerik
    text = re.sub(r'[^\w\s\.,\?!\"\']', ' ', text)

    # normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()
    
    # lowercase
    text = text.lower()

    return text

In [5]:
df['clean_text'] = df['content'].progress_apply(clean_news_bertopic)
print("Cleaning selesai")

100%|██████████| 234/234 [00:00<00:00, 2175.63it/s]

Cleaning selesai


In [6]:
minimal_stopwords = {
    'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'pada', 'dengan',
    'ini', 'itu', 'adalah', 'merupakan', 'juga', 'ada', 'oleh',
    'karena', 'atau', 'tidak', 'dalam', 'akan', 'telah', 'sudah',
    'bisa', 'harus', 'saat', 'sebagai', 'secara', 'tersebut',
    'saya', 'kita', 'kami', 'mereka', 'dia', 'anda',
    'ya', 'saja', 'lagi', 'maka', 'sehingga', 'namun', 'tetapi',
    'hal', 'sebuah', 'sangat', 'sekali', 'hanya', 'antara',
    'setelah', 'sebelum', 'kapan', 'dimana', 'berapa',
    'bagaimana', 'mengapa', 'apakah', 'jika', 'kalau',
}

def remove_minimal_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w not in minimal_stopwords]
    return ' '.join(filtered)

df['final_bertopic'] = df['clean_text'].progress_apply(remove_minimal_stopwords)
print("Stopword removal minimal selesai")

# Filter teks yang terlalu pendek (noise)
min_len = 30
initial = len(df)
df = df[df['final_bertopic'].str.len() >= min_len]
df.reset_index(drop=True, inplace=True)
print(f"Teks pendek (< {min_len} karakter) dihapus: {initial - len(df)} dokumen")
print(f"Total dokumen siap BERTopic: {len(df)}")

100%|██████████| 234/234 [00:00<00:00, 19510.33it/s]

Stopword removal minimal selesai
Teks pendek (< 30 karakter) dihapus: 0 dokumen
Total dokumen siap BERTopic: 234


In [7]:
print("=" * 60)
print("LAPORAN KUALITAS PREPROCESSING")
print("=" * 60)

df['orig_len'] = df['content'].str.len()
df['final_len'] = df['final_bertopic'].str.len()
df['orig_words'] = df['content'].apply(lambda x: len(str(x).split()))
df['final_words'] = df['final_bertopic'].apply(lambda x: len(str(x).split()))

print(f"\n Rata-rata panjang teks:")
print(f"   Original: {df['orig_len'].mean():.1f} karakter")
print(f"   Final:    {df['final_len'].mean():.1f} karakter")
print(f"   Reduksi:  {(1 - df['final_len'].mean()/df['orig_len'].mean())*100:.1f}%")

print(f"\n Rata-rata jumlah kata:")
print(f"   Original: {df['orig_words'].mean():.1f}")
print(f"   Final:    {df['final_words'].mean():.1f}")
print(f"   Reduksi:  {(1 - df['final_words'].mean()/df['orig_words'].mean())*100:.1f}%")

LAPORAN KUALITAS PREPROCESSING

 Rata-rata panjang teks:
   Original: 3042.1 karakter
   Final:    2513.5 karakter
   Reduksi:  17.4%

 Rata-rata jumlah kata:
   Original: 416.6
   Final:    331.2
   Reduksi:  20.5%


In [10]:
print("\n" + "=" * 60)
print("CONTOH HASIL PREPROCESSING")
print("=" * 60)

for i in range(min(2, len(df))):
    print(f"\n BERITA {i+1}:")
    print("\nORIGINAL (300 karakter pertama):")
    print(df['content'].iloc[i][:300] + "...")
    
    print("\nFINAL (stopwords dihapus):")
    print(df['final_bertopic'].iloc[i][:300] + "...")


CONTOH HASIL PREPROCESSING

 BERITA 1:

ORIGINAL (300 karakter pertama):
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,7...

FINAL (stopwords dihapus):
korps pemberantasan tindak pidana korupsi kortas tipikor polri mengungkap kasus dugaan korupsi ditjen energi baru terbarukan konservasi energi ebtke kementerian esdm terkait pengadaan penerangan jalan umum tenaga surya pjuts . polri menyebut kerugian kasus rp 19.522.256.578,74. dirtindak kortastipid...

 BERITA 2:

ORIGINAL (300 karakter pertama):
Sekretaris Jenderal (Sekjen) Partai Golkar, Sarmuji, menjelaskan alasan Musa Rajekshah (Ijeck) dicopot dari jabatannya sebagai ketua DPD Partai Golkar Provinsi Sumatera Utara. Sarmuji mengklaim pergantian Ijeck hanya untuk kepentingan musyawara

In [11]:
# Simpan hasil
output_file = "cleanData/berita_bertopic_ready.csv"
df[['content', 'final_bertopic']].to_csv(output_file, index=False, encoding='utf-8')
print(f"Data siap BERTopic disimpan ke: {output_file}")
print(f"Jumlah dokumen: {len(df)}")

Data siap BERTopic disimpan ke: cleanData/berita_bertopic_ready.csv
Jumlah dokumen: 234
